| Feature          | FAISS                          | Chroma                             | Pinecone                                             |
| ---------------- | ------------------------------ | ---------------------------------- | ---------------------------------------------------- |
| Product type     | Vector-search library          | Vector database                    | Managed vector database                              |
| Runs locally     | Yes                            | Yes                                | No normal local database mode                        |
| Managed cloud    | No                             | Yes                                | Yes, primary mode                                    |
| Vector index     | Directly controlled            | Database-managed                   | Fully managed                                        |
| Documents        | LangChain/external store       | Native collection record           | Metadata/document fields through records/integration |
| Metadata         | External/wrapper               | Native                             | Native                                               |
| Metadata filters | Not native FAISS DB filtering  | Native                             | Native                                               |
| CRUD             | Limited/index-dependent        | Native                             | Native                                               |
| Persistence      | Manual save/load               | Automatic persistent client/server | Managed                                              |
| Collections      | No native collection primitive | Native collections                 | Indexes and namespaces                               |
| Scaling          | You design it                  | Local/server/cloud options         | Automatically managed infrastructure                 |
| Server API       | Build yourself                 | Available                          | Built-in                                             |
| GPU/index tuning | Strong direct control          | Abstracted                         | Fully abstracted                                     |
| Cost             | Infrastructure only            | Local free/cloud paid              | Cloud usage-based                                    |
| Best for         | Research/local/custom ANN      | Local RAG and flexible deployments | Production cloud-scale retrieval                     |


## Import Libraries

In [ ]:
import os
import time
from uuid import uuid4
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is missing")

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

dimension = len(
    embeddings.embed_query("dimension check")
)
print("Embedding dimension:", dimension)

Embedding dimension: 3072


In [5]:
model = ChatOpenAI(model="gpt-4o-mini")

## Setting Up Pinecone

In [ ]:
pc = Pinecone(
    api_key=PINECONE_API_KEY
)

In [8]:
index_name = "langchain-llama-index"

In [9]:
#### CREATE INDEX IF IT DOESN'T EXIST ####
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )

In [10]:
description = pc.describe_index(index_name)
description.status["ready"]

True

In [11]:
#### CONNECT TO THE INDEX ####

index = pc.Index(index_name)

In [13]:
vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

## Creating Documents

In [14]:
documents = [
    Document(
        page_content=(
            "I had chocolate chip pancakes and "
            "scrambled eggs for breakfast this morning."
        ),
        metadata={"source": "tweet"},
    ),
    Document(
        page_content=(
            "The weather forecast for tomorrow is cloudy "
            "and overcast, with a high of 62 degrees."
        ),
        metadata={"source": "news"},
    ),
    Document(
        page_content=(
            "Building an exciting new project with "
            "LangChain - come check it out!"
        ),
        metadata={"source": "tweet"},
    ),
    Document(
        page_content=(
            "Robbers broke into the city bank and "
            "stole $1 million in cash."
        ),
        metadata={"source": "news"},
    ),
    Document(
        page_content=(
            "LangGraph is the best framework for building "
            "stateful, agentic applications!"
        ),
        metadata={"source": "tweet"},
    ),
]

In [15]:
ids = [
    str(uuid4())
    for _ in documents
]


## Adding Documents

In [16]:
inserted_ids = vector_store.add_documents(
    documents=documents,
    ids=ids
)

print("Inserted IDs:", inserted_ids)

Inserted IDs: ['d89d3ae6-43e5-4a94-93c3-79ec5722182f', '142d608c-c2d8-408b-9a1c-ec59b965bb3f', '808bc257-0782-46c1-ac1a-1b53c09f66ce', '622d935c-8387-439f-8e79-9813ab88d6cd', 'f0020a3c-239b-4225-b432-4738f4a4959a']


## Similarity Search with Metadata Filter

In [17]:
results = vector_store.similarity_search(
    query=(
        "LangChain provides abstractions "
        "for working with LLMs"
    ),
    k=2,
    filter={
        "source": "tweet"
    }
)

for result in results:
    print(result.page_content)
    print(result.metadata)

Building an exciting new project with LangChain - come check it out!
{'source': 'tweet'}
LangGraph is the best framework for building stateful, agentic applications!
{'source': 'tweet'}


## Retriever with threshold and filter

In [18]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 1,
        "score_threshold": 0.4,
        "filter": {
            "source": "news"
        }
    }
)

retrieved_docs = retriever.invoke(
    "Stealing money from a bank is a crime"
)

print(retrieved_docs)

[Document(id='622d935c-8387-439f-8e79-9813ab88d6cd', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]


## Deleting a Document

In [19]:
vector_store.delete(
    ids=[ids[-1]]
)